# 1. Data Cleaning & Preprocessing


> **Goal:**  This notebook handles the initial raw data loading, type conversion, duplicates removal, and basic filtering.
### Data Selection & Subsetting

To ensure efficient processing, we work with a representative subset of the data:

* **Raw Dataset:** 29,222,434 rows
* **Project Subset:** 1,000,000 rows (First 1M sorted by lp_id)
* **Rationale:** Computational efficiency and processing speed.
* **Scope:** 1,117 unique charging points included.

### Output
| Year | Filename | Row Count |
| :--- | :--- | :--- |
| **Total** | `df_subset_1M.csv` | **1,000,000** |
| 2018 | `data_2018.csv` | 5,050 |
| 2019 | `data_2019.csv` | 23,078 |
| 2020 | `data_2020.csv` | 52,087 |
| 2021 | `data_2021.csv` | 128,237 |
| 2022 | `data_2022.csv` | 215,112 |
| 2023 | `data_2023.csv` | 262,401 |
| 2024 | `data_2024.csv` | 314,035 |



In [4]:
import pandas as pd

## 1. Load Data & Create Subset



We sort the Data after lp_id (Charingpoint ID) and then only take the first 1.000.000 rows. And also save this subset for reproducability and waittime in 'data/interim/df_subset_1M.csv'

In [5]:
# Define paths
RAW_DATA_PATH = '../data/raw/df_lv.csv'
OUTPUT_FOLDER = '../data/interim/'
SUBSET_PATH =  '../data/interim/df_subset_1M.csv'

try:
    # 1. Try to open the file to see if it exists
    with open(SUBSET_PATH, 'r') as f:
        print(f"Subset found at {SUBSET_PATH}. Loading existing file...")
    
    # If the block above didn't fail, load the subset
    df_subset = pd.read_csv(SUBSET_PATH)

except FileNotFoundError:
    # 2. This runs only if the file does NOT exist
    print("Subset not found. Processing raw data...")
    
    # Load raw data
    df = pd.read_csv(RAW_DATA_PATH, sep=';')
    print(f"Initial shape: {df.shape}")
    
    # Sort and slice
    df = df.sort_values('lp_id').reset_index(drop=True)
    df_subset = df.iloc[:1000000].copy()
    
    # Save (Note: to_csv will create the file, but it cannot create 
    # the folder 'interim' if it doesn't already exist)
    df_subset.to_csv(SUBSET_PATH, index=False)
    print(f"Saved subset to: {SUBSET_PATH}")

# 3. Final Diagnostics
print(f"Current subset row count: {len(df_subset)}")
print(f"Unique lp_id count: {df_subset.lp_id.nunique()}")

Subset not found. Processing raw data...
Initial shape: (29222434, 10)
Saved subset to: ../data/interim/df_subset_1M.csv
Current subset row count: 1000000
Unique lp_id count: 1117


## 2. Save Data

Saving the Data for each year in the dataset. 

In [7]:
# Convert DateTime columns
print("Converting datetime columns...")
df_subset['beginn'] = pd.to_datetime(df_subset['beginn'])
df_subset['ende'] = pd.to_datetime(df_subset['ende'])

yearly_data = {}
for year in range(2018, 2025):
    yearly_data[year] = df_subset[df_subset['beginn'].dt.year == year]


for year, data in yearly_data.items():
    # Construct the filename using f-string concatenation
    file_path = f"{OUTPUT_FOLDER}data_{year}.csv"
    
    # Save to CSV
    data.to_csv(file_path, index=False)
    
    print(f"Saved: {file_path} ({len(data)} rows)")

Converting datetime columns...
Saved: ../data/interim/data_2018.csv (5050 rows)
Saved: ../data/interim/data_2019.csv (23078 rows)
Saved: ../data/interim/data_2020.csv (52087 rows)
Saved: ../data/interim/data_2021.csv (128237 rows)
Saved: ../data/interim/data_2022.csv (215112 rows)
Saved: ../data/interim/data_2023.csv (262401 rows)
Saved: ../data/interim/data_2024.csv (314035 rows)
